# Download Stage E output to your PC
from google.colab import files
import config

stage_e_file = config.stage_e_dir() / 'stage_e_validated_output.json'
if stage_e_file.exists():
    files.download(str(stage_e_file))
    print('[OK] Downloaded: stage_e_validated_output.json')
    print('     (intermediate output - Stage F and G follow)')
else:
    print('[!] File not found. Run Stage E cell first.')

## Step 1: Install all dependencies

Run this cell once. Do **not** restart the runtime after it finishes.

In [ ]:
import os
os.makedirs('/content/pip_logs', exist_ok=True)

print('[*] Installing Stage A+B dependencies (docling, pymupdf, pdfplumber)...')
!pip install -q docling pymupdf pdfplumber 2>>/content/pip_logs/install.log

print('[*] Installing Stage C+D+E dependencies (torch, transformers, sentence-transformers, faiss)...')
!pip install -q torch transformers accelerate sentencepiece sentence-transformers faiss-cpu 2>>/content/pip_logs/install.log

print('[*] Installing utilities (nltk, pydantic, huggingface_hub)...')
!pip install -q nltk pydantic huggingface_hub 2>>/content/pip_logs/install.log

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print('[OK] All dependencies installed!')

## Step 2: Verify GPU

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU not enabled! Go to: Runtime → Change runtime type → T4 GPU → Save, then reconnect.')
print(f'[OK] GPU ready: {torch.cuda.get_device_name(0)}')
print(f'     VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Step 3: Login to Hugging Face

Required for LLaMA 3.2 (used in Stage D disambiguation and Stage E extraction).
1. Get your token at https://huggingface.co/settings/tokens
2. Accept the LLaMA license at https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct
3. Paste the token when prompted below (or save it as a Colab Secret named `HF_TOKEN`)

In [ ]:
import getpass
from huggingface_hub import login

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print('[OK] Logged in via Colab Secrets (HF_TOKEN)')
except Exception:
    hf_token = getpass.getpass('Paste your Hugging Face token: ')
    login(token=hf_token)
    print('[OK] Logged in to Hugging Face')

## Step 4: Upload project ZIP

Upload `Thesis_llama_colab.zip` (created by running `python create_colab_zip.py` on your PC).

In [ ]:
import sys
import os
import zipfile
from pathlib import Path
from google.colab import files

print('[*] Upload Thesis_llama_colab.zip ...')
uploaded = files.upload()

for fn in uploaded:
    if fn.endswith('.zip'):
        with zipfile.ZipFile(fn, 'r') as z:
            z.extractall('/content')
        print(f'[OK] Extracted: {fn}')
        break

# Auto-detect project root (look for pipeline/ folder)
project_dir = '/content'
for d in os.listdir('/content'):
    p = Path('/content') / d
    if p.is_dir() and (p / 'pipeline').exists():
        project_dir = str(p)
        break

if not (Path(project_dir) / 'pipeline').exists():
    raise RuntimeError('pipeline/ folder not found after extraction. Make sure you uploaded the correct ZIP.')

sys.path.insert(0, project_dir)
os.chdir(project_dir)
print(f'[OK] Project root: {project_dir}')

import config
print(f'[OK] Config loaded. Versions: A={config.DEFAULT_STAGE_A_VERSION}, B={config.DEFAULT_STAGE_B_VERSION}, C={config.DEFAULT_STAGE_C_VERSION}, D={config.DEFAULT_STAGE_D_VERSION}, E={config.DEFAULT_STAGE_E_VERSION}')

## Step 5: Upload your PDF file(s)

Upload all medical guideline PDFs. They will be placed in the `input/` folder.

In [ ]:
import shutil
from google.colab import files
from pathlib import Path

input_dir = Path(project_dir) / 'input'
input_dir.mkdir(parents=True, exist_ok=True)

print('[*] Upload your PDF file(s)...')
uploaded_pdfs = files.upload()

for fname in uploaded_pdfs:
    dest = input_dir / fname
    src = Path(fname)
    if src.exists():
        shutil.move(str(src), str(dest))
    else:
        with open(dest, 'wb') as f:
            f.write(uploaded_pdfs[fname])
    print(f'  [+] Saved: {dest.name}')

all_pdfs = list(input_dir.glob('*.pdf'))
if not all_pdfs:
    raise FileNotFoundError('No PDFs found! Run this cell again and select your PDF file(s).')
print(f'[OK] {len(all_pdfs)} PDF(s) ready.')

## Step 6: Upload UMLS.csv

Upload `UMLS.csv` from your local `input/` folder. This is ~213 MB — it may take a minute to upload.

In [ ]:
import shutil
from google.colab import files
from pathlib import Path

input_dir = Path(project_dir) / 'input'
UMLS_PATH = input_dir / 'UMLS.csv'

if UMLS_PATH.exists():
    print(f'[OK] UMLS.csv already present ({UMLS_PATH.stat().st_size / 1e6:.1f} MB). Skipping upload.')
else:
    print('[*] Upload UMLS.csv (from your input/ folder, ~213 MB)...')
    uploaded_umls = files.upload()
    for fname in uploaded_umls:
        if 'UMLS' in fname and fname.endswith('.csv'):
            dest = UMLS_PATH
            src = Path(fname)
            if src.exists():
                shutil.move(str(src), str(dest))
            else:
                with open(dest, 'wb') as f:
                    f.write(uploaded_umls[fname])
            print(f'[OK] UMLS.csv saved ({dest.stat().st_size / 1e6:.1f} MB)')
            break
    if not UMLS_PATH.exists():
        raise FileNotFoundError('UMLS.csv not saved. Run this cell again and select UMLS.csv.')

---
## Stage A v2 — PDF Extraction (Docling)

Reads your PDFs page by page using Docling, extracts clean text and tables.
Skips first 3 pages (cover/TOC) and last 5 pages (references).

**Expected time:** 5–20 min depending on PDF length.

In [ ]:
import sys
import os
sys.path.insert(0, project_dir)
os.chdir(project_dir)

import config
from pipeline.data import PDFGuidelines
from pipeline.transforms import ExtractTextV2

STAGE_A_DIR = config.stage_a_dir()
STAGE_A_DIR.mkdir(parents=True, exist_ok=True)

pdf_guidelines = PDFGuidelines(pdf_dir=str(Path(project_dir) / 'input'))
print(f'[*] Found {pdf_guidelines.count()} PDF(s): {[p.name for p in pdf_guidelines.get_files()]}')

extractor = ExtractTextV2(
    skip_first_pages=3,
    skip_last_pages=5,
    stage_output_dir=str(STAGE_A_DIR),
)
print('[*] Running Stage A v2 (Docling extraction — please wait)...')
stage_a_raw_text = extractor.transform(pdf_guidelines)

print(f'[OK] Stage A v2 complete: {stage_a_raw_text.count()} pages extracted → {STAGE_A_DIR}')

In [ ]:
# Download Stage A outputs to your PC
from google.colab import files
import config

STAGE_A_DIR = config.stage_a_dir()
for fname in ['text.json', 'tables.json']:
    fpath = STAGE_A_DIR / fname
    if fpath.exists():
        files.download(str(fpath))
        print(f'[OK] Downloaded: {fname}')
    else:
        print(f'[!] Not found: {fname}')

---
## Stage B v2 — Text Chunking + Table Triples

Splits extracted text into clean chunks (max ~400 chars).
v2 filters out chunks that are just table-body content already captured as SPO triples.

**Expected time:** 1–3 min.

In [ ]:
import sys
import os
sys.path.insert(0, project_dir)
os.chdir(project_dir)

import config
from pipeline.stage_io import load_stage_a_output
from pipeline.models import ParsingRules
from pipeline.transforms import ContentPreparation

STAGE_A_DIR = config.stage_a_dir()
STAGE_B_DIR = config.stage_b_dir()

# Read Stage A output from disk (already there from Stage A cell above)
raw_text = load_stage_a_output(STAGE_A_DIR)
if raw_text is None:
    raise FileNotFoundError('Stage A output not found. Run Stage A cell first.')
print(f'[*] Loaded Stage A: {raw_text.count()} pages')

parsing_rules = ParsingRules()
content_prep = ContentPreparation(
    parsing_rules=parsing_rules,
    min_chars=40,
    stage_output_dir=str(STAGE_B_DIR),
    stage_b_version='v2',
)
print('[*] Running Stage B v2 (chunking + table triples)...')
stage_b_result = content_prep.transform(raw_text)

text_chunks = stage_b_result.text_chunks
table_triples = stage_b_result.table_triples
print(f'[OK] Stage B v2 complete: {text_chunks.count()} text chunks, {len(table_triples)} table triples → {STAGE_B_DIR}')

In [ ]:
# Download Stage B outputs to your PC
from google.colab import files
import config

STAGE_B_DIR = config.stage_b_dir()
for fname in ['stage_b_text_chunks.json', 'stage_b_table_triples.json']:
    fpath = STAGE_B_DIR / fname
    if fpath.exists():
        files.download(str(fpath))
        print(f'[OK] Downloaded: {fname}')
    else:
        print(f'[!] Not found: {fname}')

---
## Stage C v2 — Named Entity Recognition (NER)

Runs the `d4data/biomedical-ner-all` model on every text chunk to find medical entities
(diseases, drugs, anatomy, etc.).

**Expected time:** 10–30 min depending on number of chunks.

In [ ]:
import sys
import os
import json
sys.path.insert(0, project_dir)
os.chdir(project_dir)

import config
from pipeline.stage_io import load_stage_b_output
from pipeline.models import NeuralModel
from pipeline.inference import RecognizeEntities

STAGE_B_DIR = config.stage_b_dir()
STAGE_C_DIR = config.stage_c_dir()
STAGE_C_DIR.mkdir(parents=True, exist_ok=True)

# Load Stage B from disk
text_chunks, table_triples_b = load_stage_b_output(STAGE_B_DIR)
print(f'[*] Loaded Stage B: {text_chunks.count()} chunks, {len(table_triples_b)} table triples')

print('[*] Loading NER model (d4data/biomedical-ner-all)...')
ner_model = NeuralModel(model_name='d4data/biomedical-ner-all')
recognizer = RecognizeEntities(neural_model=ner_model, min_score=0.55, acronym_file=None)

print('[*] Running Stage C v2 (NER on all chunks)...')
stage_c_result = recognizer.infer(text_chunks)
table_triples_c = recognizer.enrich_triples_with_entities(table_triples_b)

# Write Stage C output
stage_c_file = STAGE_C_DIR / 'stage_c_statements_with_entities.json'
stage_c_data = {
    'metadata': {
        'stage': 'c',
        'description': 'Statements with medical entities (NER) and table triples from Stage B',
        'total_statements': stage_c_result.count(),
        'total_entities': stage_c_result.get_entity_count(),
        'total_table_triples': len(table_triples_c),
        'neural_model': 'd4data/biomedical-ner-all',
        'min_ner_score': 0.55,
    },
    'statements': stage_c_result.get_all(),
    'table_triples': table_triples_c,
}
stage_c_file.write_text(json.dumps(stage_c_data, indent=2, ensure_ascii=False), encoding='utf-8')

print(f'[OK] Stage C v2 complete: {stage_c_result.count()} statements, {stage_c_result.get_entity_count()} entities → {stage_c_file}')

# Free NER model memory before Stage D
import torch
del ner_model, recognizer
torch.cuda.empty_cache()
print('[OK] NER model freed from GPU memory')

In [ ]:
# Download Stage C output to your PC
from google.colab import files
import config

stage_c_file = config.stage_c_dir() / 'stage_c_statements_with_entities.json'
if stage_c_file.exists():
    files.download(str(stage_c_file))
    print('[OK] Downloaded: stage_c_statements_with_entities.json')
else:
    print('[!] File not found. Run Stage C cell first.')

---
## Stage D v2 — UMLS Entity Linking (SapBERT bi-encoder + LLaMA disambiguation)

### Sub-step D1: Build the UMLS FAISS index

Encodes up to 200,000 UMLS concepts using SapBERT and stores them in a FAISS index.
This is the slowest step — **expect 15–30 minutes**.
The index is saved to disk so it only needs to be built once.

In [ ]:
import sys
import importlib
sys.path.insert(0, project_dir)

import pipeline.models.entities_linker
importlib.reload(pipeline.models.entities_linker)
from pipeline.models.entities_linker import build_biencoder_index
from pathlib import Path
import config

UMLS_PATH = Path(project_dir) / 'input' / 'UMLS.csv'
INDEX_DIR = Path('/content/stage_d_v2_index')  # saved to /content so it survives between cells
BIENCODER_MODEL = 'cambridgeltl/SapBERT-from-PubMedBERT-fulltext'
MAX_CONCEPTS = 200000  # increase or set to None for full UMLS (slower)

if not UMLS_PATH.exists():
    raise FileNotFoundError(f'UMLS.csv not found at {UMLS_PATH}. Run Step 6 first.')

if INDEX_DIR.exists() and list(INDEX_DIR.iterdir()):
    print(f'[OK] FAISS index already built at {INDEX_DIR}. Skipping index build.')
else:
    print(f'[*] Building UMLS FAISS index (up to {MAX_CONCEPTS:,} concepts — this takes 15–30 min)...')
    n, _ = build_biencoder_index(
        str(UMLS_PATH),
        BIENCODER_MODEL,
        str(INDEX_DIR),
        batch_size=50000,
        max_concepts=MAX_CONCEPTS,
    )
    print(f'[OK] FAISS index built: {n:,} concepts indexed in {INDEX_DIR}')

### Sub-step D2: Load LLaMA for disambiguation (optional but recommended)

LLaMA 3.2 3B is used to disambiguate between UMLS candidate matches.
Set `USE_LLM_DISAMBIGUATION = False` if you run out of GPU memory or want to skip this.

**The LLaMA model loaded here is reused for Stage E** — no double-loading.

In [ ]:
USE_LLM_DISAMBIGUATION = True  # Set to False to skip LLaMA disambiguation
LLAMA_MODEL = 'meta-llama/Llama-3.2-3B-Instruct'

STAGE_E_VALIDATION_MODEL = None  # Will be set below; reused for Stage E
DISAMBIGUATOR = None

if USE_LLM_DISAMBIGUATION:
    import sys
    sys.path.insert(0, project_dir)
    from pipeline.models import ValidationModel, make_llm_disambiguator

    print(f'[*] Loading LLaMA ({LLAMA_MODEL}) — this may take a few minutes...')
    STAGE_E_VALIDATION_MODEL = ValidationModel(model_name=LLAMA_MODEL)
    DISAMBIGUATOR = make_llm_disambiguator(STAGE_E_VALIDATION_MODEL.run_inference, max_new_tokens=50)
    print('[OK] LLaMA loaded and ready (will be reused for Stage E — no need to reload)')
else:
    print('[*] LLM disambiguation disabled. Stage D will use type-reranking only.')
    print('    LLaMA will still be loaded fresh for Stage E.')

### Sub-step D3: Run entity linking

In [ ]:
import sys
import os
import json
sys.path.insert(0, project_dir)
os.chdir(project_dir)

import importlib
import pipeline.models.entities_linker
importlib.reload(pipeline.models.entities_linker)

import config
from pathlib import Path
from pipeline.data import StatementsWithMedicalEntities, CandidateStatements
from pipeline.models.entities_linker import BiEncoderLinker

STAGE_C_FILE = config.stage_c_dir() / 'stage_c_statements_with_entities.json'
STAGE_D_DIR = config.stage_d_dir()
STAGE_D_DIR.mkdir(parents=True, exist_ok=True)

# Load Stage C from disk
print('[*] Loading Stage C output...')
with open(STAGE_C_FILE, 'r', encoding='utf-8') as f:
    stage_c_data = json.load(f)

statements_with_entities = StatementsWithMedicalEntities()
for stmt in stage_c_data['statements']:
    statements_with_entities.add_statement(stmt)
table_triples_raw = stage_c_data.get('table_triples', [])
print(f'    Statements: {statements_with_entities.count()}, Table triples: {len(table_triples_raw)}')

print('[*] Loading bi-encoder linker (SapBERT + FAISS index)...')
linker = BiEncoderLinker(
    model_name=BIENCODER_MODEL,
    index_dir=str(INDEX_DIR),
    top_k=16,
    min_link_score=0.72,
    use_type_rerank=True,
    prefer_shorter_concept=True,
    disambiguator=DISAMBIGUATOR,
)
print(f'    {linker.get_umls_stats()}')
if USE_LLM_DISAMBIGUATION and DISAMBIGUATOR:
    print('    LLaMA disambiguation: ON')

print('[*] Linking entities in statements...')
result = CandidateStatements()
for i, stmt in enumerate(statements_with_entities.get_all()):
    text = stmt.get('text', '')
    raw_entities = stmt.get('entities', [])
    linked = linker.link_entities(entities=raw_entities, context_text=text)
    result.add_statement({
        'chunk_id': stmt.get('chunk_id'),
        'page': stmt.get('page'),
        'source': stmt.get('source', ''),
        'text': text,
        'original_text': stmt.get('original_text', text),
        'entities': linked,
    })
    if (i + 1) % 200 == 0:
        print(f'    Linked {i + 1}/{statements_with_entities.count()} statements')

print('[*] Linking entities in table triples...')
table_triples_enriched = []
for triple in table_triples_raw:
    out = dict(triple)
    entities = triple.get('entities', [])
    if entities:
        triple_context = ' '.join(
            part.strip() for part in (
                triple.get('subject', ''),
                triple.get('predicate', ''),
                triple.get('object', ''),
            ) if isinstance(part, str) and part.strip()
        )
        linker_input = [{'text': e.get('text', ''), 'label': e.get('label', '')} for e in entities]
        for i, e in enumerate(entities):
            if i < len(linker_input) and 'score' in e:
                linker_input[i]['score'] = e['score']
        out['entities'] = linker.link_entities(linker_input, context_text=triple_context)
    table_triples_enriched.append(out)

# Write Stage D output
stage_d_file = STAGE_D_DIR / 'stage_d_candidate_statements.json'
stage_d_output = {
    'metadata': {
        'stage': 'd',
        'stage_d_version': 'v2',
        'description': 'Candidate statements and table triples with UMLS-linked entities (bi-encoder)',
        'total_statements': result.count(),
        'total_candidates': result.count_candidates(),
        'total_table_triples': len(table_triples_enriched),
        'umls_linking': True,
    },
    'statements': result.get_all(),
    'table_triples': table_triples_enriched,
}
with open(stage_d_file, 'w', encoding='utf-8') as f:
    json.dump(stage_d_output, f, indent=2, ensure_ascii=False)

print(f'[OK] Stage D v2 complete: {result.count()} statements → {stage_d_file}')

# Free linker/SapBERT from GPU (keep LLaMA alive for Stage E)
import torch
del linker
torch.cuda.empty_cache()
print('[OK] SapBERT linker freed. LLaMA kept alive for Stage E.')

In [ ]:
# Download Stage D output to your PC
from google.colab import files
import config

stage_d_file = config.stage_d_dir() / 'stage_d_candidate_statements.json'
if stage_d_file.exists():
    files.download(str(stage_d_file))
    print('[OK] Downloaded: stage_d_candidate_statements.json')
    print('     (This is a complete intermediate output — useful if Stage E fails)')
else:
    print('[!] File not found. Run Stage D linking cell first.')

---
## Stage E — LLM Factual Statement Extraction (FINAL OUTPUT)

LLaMA 3.2 reads every candidate statement from Stage D and extracts structured
factual triples: `(subject, predicate, object, exception, duration)`.
Also processes table triples through the LLM.

**Expected time:** 30–90 min depending on number of statements.

**Output:** `stage_e_validated_output.json` — your final structured knowledge base.

In [ ]:
import sys
import os
import json
import time
sys.path.insert(0, project_dir)
os.chdir(project_dir)

import config
from pathlib import Path
from pipeline.data import CandidateStatements, ValidatedFactsAndQualifiers
from pipeline.models import ValidationModel
from pipeline.inference import Validate

STAGE_D_FILE = config.stage_d_dir() / 'stage_d_candidate_statements.json'
STAGE_E_DIR = config.stage_e_dir()
STAGE_E_DIR.mkdir(parents=True, exist_ok=True)
STAGE_E_FILE = STAGE_E_DIR / 'stage_e_validated_output.json'

LLAMA_MODEL = 'meta-llama/Llama-3.2-3B-Instruct'
BATCH_SIZE = 4
MAX_EXTRACTION_TOKENS = 400

# Load Stage D from disk
print('[*] Loading Stage D output...')
with open(STAGE_D_FILE, 'r', encoding='utf-8') as f:
    stage_d_data = json.load(f)

candidate_statements = CandidateStatements()
for stmt in stage_d_data.get('statements', []):
    candidate_statements.add_statement(stmt)
table_triples_d = stage_d_data.get('table_triples', [])
print(f'    Loaded {candidate_statements.count()} statements, {len(table_triples_d)} table triples')

# Reuse LLaMA from Stage D if already loaded; otherwise load fresh
if 'STAGE_E_VALIDATION_MODEL' in dir() and STAGE_E_VALIDATION_MODEL is not None:
    print('[*] Reusing LLaMA model already loaded from Stage D (no reload needed)...')
    validation_model = STAGE_E_VALIDATION_MODEL
else:
    print(f'[*] Loading LLaMA ({LLAMA_MODEL}) for Stage E...')
    validation_model = ValidationModel(model_name=LLAMA_MODEL)

validator = Validate(
    validation_model=validation_model,
    batch_size=BATCH_SIZE,
    max_new_tokens=MAX_EXTRACTION_TOKENS,
)

start = time.time()
print('[*] Stage E: Extracting factual triples from statements...')
validated_facts = validator.validate(candidate_statements)

if table_triples_d:
    print(f'[*] Stage E: Processing {len(table_triples_d)} table triples through LLM...')
    validator.validate_table_triples(validated_facts, table_triples_d)

elapsed = time.time() - start

# Write Stage E output
stage_e_output = {
    'metadata': {
        'stage': 'e',
        'description': 'Extracted factual statements for expert validation',
        'total_statements': validated_facts.count(),
        'extraction_model': LLAMA_MODEL,
        'table_triples_through_llm': len(table_triples_d),
    },
    'validated_statements': validated_facts.get_all(),
}
with open(STAGE_E_FILE, 'w', encoding='utf-8') as f:
    json.dump(stage_e_output, f, indent=2, ensure_ascii=False)

print(f'\n[SUCCESS] Stage E complete in {elapsed / 60:.1f} min!')
print(f'          Total factual statements: {validated_facts.count()}')
print(f'          Output: {STAGE_E_FILE}')

In [ ]:
# Download Stage E final output to your PC
from google.colab import files
import config

stage_e_file = config.stage_e_dir() / 'stage_e_validated_output.json'
if stage_e_file.exists():
    files.download(str(stage_e_file))
    print('[OK] Downloaded: stage_e_validated_output.json')
    print('     *** This is your FINAL OUTPUT — all extracted factual statements! ***')
else:
    print('[!] File not found. Run Stage E cell first.')

---
## Stage F — SHACL Constraint Extraction

Reads every factual statement from Stage E that has medical entities with UMLS CUIs.
Uses LLaMA (already loaded — no reload needed) to extract formal SHACL-style constraints:

- **Dosage ranges** — min/max dose and units (e.g. `dosage: range 2.5mg – 40mg`)
- **Frequency** — how often (e.g. `frequency: enum [daily, twice daily]`)
- **Contraindications** — NOT conditions (e.g. `contraindication: NOT pregnancy`)
- **Patient conditions** — AND/OR applicability rules
- **Monitoring** — temporal check requirements (e.g. `monitoring: every 3 months`)

**Expected time:** 20–60 min (one LLaMA call per medically-relevant statement).

**Output:** `stage_f_shacl_constraints.json`

In [ ]:
import sys, os, json, time
sys.path.insert(0, project_dir)
os.chdir(project_dir)

import config
from pathlib import Path
from pipeline.data import ValidatedFactsAndQualifiers
from pipeline.models import ShaclExtractor
from pipeline.inference import ExtractShacl

STAGE_E_FILE = config.stage_e_dir() / 'stage_e_validated_output.json'
STAGE_F_DIR  = config.stage_f_dir()
STAGE_F_DIR.mkdir(parents=True, exist_ok=True)
STAGE_F_FILE = STAGE_F_DIR / 'stage_f_shacl_constraints.json'

print('[*] Loading Stage E output...')
with open(STAGE_E_FILE, 'r', encoding='utf-8') as f:
    stage_e_data = json.load(f)
validated_facts_f = ValidatedFactsAndQualifiers()
for stmt in stage_e_data.get('validated_statements', []):
    validated_facts_f.add_validated(stmt)
print(f'    Loaded {validated_facts_f.count()} statements')

# Reuse LLaMA already in memory from Stage E/D
if 'STAGE_E_VALIDATION_MODEL' in dir() and STAGE_E_VALIDATION_MODEL is not None:
    print('[*] Reusing LLaMA from Stage E/D (no reload)...')
    shacl_llm = STAGE_E_VALIDATION_MODEL
else:
    from pipeline.models import ValidationModel
    print('[*] Loading LLaMA for Stage F...')
    shacl_llm = ValidationModel(model_name='meta-llama/Llama-3.2-3B-Instruct')

shacl_extractor = ShaclExtractor(validation_model=shacl_llm, max_new_tokens=350)
extractor_f = ExtractShacl(shacl_extractor=shacl_extractor)

start_f = time.time()
print('[*] Running Stage F (SHACL constraint extraction)...')
shacl_constraints = extractor_f.extract(validated_facts_f)
elapsed_f = time.time() - start_f

stage_f_output = {
    'metadata': {
        'stage': 'f',
        'description': 'SHACL-style constraints extracted from Stage E factual statements',
        'total_constraints': shacl_constraints.count(),
        'extraction_model': 'meta-llama/Llama-3.2-3B-Instruct',
        'focus': 'medications, dosage, frequency, contraindications, monitoring',
    },
    'constraints': shacl_constraints.get_all(),
}
with open(STAGE_F_FILE, 'w', encoding='utf-8') as f:
    json.dump(stage_f_output, f, indent=2, ensure_ascii=False)

print(f'[OK] Stage F complete in {elapsed_f / 60:.1f} min -- {shacl_constraints.count()} constraints extracted')

In [ ]:
# Download Stage F output to your PC
from google.colab import files
import config
stage_f_file = config.stage_f_dir() / 'stage_f_shacl_constraints.json'
if stage_f_file.exists():
    files.download(str(stage_f_file))
    print('[OK] Downloaded: stage_f_shacl_constraints.json')
else:
    print('[!] File not found. Run Stage F cell first.')

---
## Stage G — Constraint Validation (TRUE FINAL OUTPUT)

Cross-validates Stage E facts against Stage F SHACL constraints.
**Rule-based — no LLM — runs in under a minute.**

Checks per fact:
- **Range violations** — numeric values vs min/max
- **Contraindication coverage** — missing exception for a NOT-condition
- **Temporal violations** — duration out of bounds
- **Cross-statement consistency** — same concept with very different values across statements

Status per fact: **VALID** | **VIOLATED** | **UNVERIFIED**

**Output:** `stage_g_validation_results.json`

In [ ]:
import sys, os, json, time
sys.path.insert(0, project_dir)
os.chdir(project_dir)

import config
from pathlib import Path
from pipeline.data import ValidatedFactsAndQualifiers
from pipeline.data.shacl_constraints import ShaclConstraints
from pipeline.inference import ValidateConstraints

STAGE_E_FILE = config.stage_e_dir() / 'stage_e_validated_output.json'
STAGE_F_FILE = config.stage_f_dir() / 'stage_f_shacl_constraints.json'
STAGE_G_DIR  = config.stage_g_dir()
STAGE_G_DIR.mkdir(parents=True, exist_ok=True)
STAGE_G_FILE = STAGE_G_DIR / 'stage_g_validation_results.json'

print('[*] Loading Stage E facts...')
with open(STAGE_E_FILE, 'r', encoding='utf-8') as f:
    stage_e_data = json.load(f)
validated_facts_g = ValidatedFactsAndQualifiers()
for stmt in stage_e_data.get('validated_statements', []):
    validated_facts_g.add_validated(stmt)
print(f'    {validated_facts_g.count()} facts loaded')

print('[*] Loading Stage F SHACL constraints...')
with open(STAGE_F_FILE, 'r', encoding='utf-8') as f:
    stage_f_data = json.load(f)
shacl_constraints_g = ShaclConstraints()
for c in stage_f_data.get('constraints', []):
    shacl_constraints_g.add_constraint(c)
print(f'    {shacl_constraints_g.count()} constraints loaded')

start_g = time.time()
print('[*] Running Stage G (rule-based validation)...')
validator_g = ValidateConstraints()
validation_results = validator_g.validate(validated_facts_g, shacl_constraints_g)
elapsed_g = time.time() - start_g

summary = validation_results.summary()

stage_g_output = {
    'metadata': {
        'stage': 'g',
        'description': 'Validation: Stage E facts cross-validated against Stage F SHACL constraints',
        'summary': summary,
    },
    'validation_results': validation_results.get_all(),
    'violations': validation_results.get_violations(),
}
with open(STAGE_G_FILE, 'w', encoding='utf-8') as f:
    json.dump(stage_g_output, f, indent=2, ensure_ascii=False)

print(f'[SUCCESS] Stage G complete in {elapsed_g:.1f}s!')
print(f'  Total   : {summary["total"]}')
print(f'  VALID   : {summary["valid"]} ({summary["valid_pct"]}%)')
print(f'  VIOLATED: {summary["violated"]} ({summary["violated_pct"]}%)')
print(f'  UNVERIF : {summary["unverified"]}')

In [ ]:
# Download Stage G - TRUE FINAL OUTPUT
from google.colab import files
import config
stage_g_file = config.stage_g_dir() / 'stage_g_validation_results.json'
if stage_g_file.exists():
    files.download(str(stage_g_file))
    print('[OK] Downloaded: stage_g_validation_results.json')
    print('     *** TRUE FINAL OUTPUT: facts + SHACL constraints + validation status ***')
else:
    print('[!] File not found. Run Stage G cell first.')